In [ ]:
# Unzip the diabetes-data.tar.Z file

import io
import tarfile
from pathlib import Path
import unlzw3

z_path = Path("../assets/dataset/raw/diabetes-data.tar.Z").resolve()
out_dir = z_path.parent

blob = z_path.read_bytes()
decompressed = unlzw3.unlzw(blob)

# PyPI readme says string; tar data must be bytes for tarfile.
if isinstance(decompressed, str):
    decompressed = decompressed.encode("latin-1")

with tarfile.open(fileobj=io.BytesIO(decompressed), mode="r:") as tf:
    tf.extractall(path=out_dir)

In [5]:
# Format txt into CSV format
import re
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../assets/dataset/Diabetes-Data").resolve()  # adjust if cwd differs
OUTPUT = DATA_DIR / "diabetes_all_patients.csv"

pattern = re.compile(r"^data-(\d+)$")

chunks = []
for path in sorted(DATA_DIR.iterdir()):
    m = pattern.match(path.name)
    if not m or not path.is_file():
        continue
    patient_id = int(m.group(1))

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["Date", "Time", "Code", "Value"],
        dtype=str,  # keeps leading zeros in Value (e.g. "009")
    )
    df.insert(0, "patientId", patient_id)
    chunks.append(df)

result = pd.concat(chunks, ignore_index=True)
result.to_csv(OUTPUT, index=False)
print(len(chunks), "patients,", len(result), "rows ->", OUTPUT)

70 patients, 29330 rows → C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\ModelingOptimization\AppliedModelingOptimization\Ahmad\assets\dataset\Diabetes-Data\diabetes_all_patients.csv


In [6]:
# Format txt into CSV format with codes
import re
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../assets/dataset/Diabetes-Data").resolve()  # adjust if cwd differs
OUTPUT = DATA_DIR / "diabetes_all_patients_with_codes.csv"

CODE_MEANINGS = {
    "33": "Regular insulin dose",
    "34": "NPH insulin dose",
    "35": "UltraLente insulin dose",
    "48": "Unspecified blood glucose measurement",
    "57": "Unspecified blood glucose measurement",
    "58": "Pre-breakfast blood glucose measurement",
    "59": "Post-breakfast blood glucose measurement",
    "60": "Pre-lunch blood glucose measurement",
    "61": "Post-lunch blood glucose measurement",
    "62": "Pre-supper blood glucose measurement",
    "63": "Post-supper blood glucose measurement",
    "64": "Pre-snack blood glucose measurement",
    "65": "Hypoglycemic symptoms",
    "66": "Typical meal ingestion",
    "67": "More-than-usual meal ingestion",
    "68": "Less-than-usual meal ingestion",
    "69": "Typical exercise activity",
    "70": "More-than-usual exercise activity",
    "71": "Less-than-usual exercise activity",
    "72": "Unspecified special event",
}

pattern = re.compile(r"^data-(\d+)$")

chunks = []
for path in sorted(DATA_DIR.iterdir()):
    m = pattern.match(path.name)
    if not m or not path.is_file():
        continue

    patient_id = int(m.group(1))

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["Date", "Time", "Code", "Value"],
        dtype=str,  # keeps leading zeros in Value (e.g. "009")
    )

    df.insert(0, "patientId", patient_id)
    df["CodeMeaning"] = df["Code"].map(CODE_MEANINGS).fillna("Unknown code")
    chunks.append(df)

result = pd.concat(chunks, ignore_index=True)
result.to_csv(OUTPUT, index=False)

print(len(chunks), "patients,", len(result), "rows ->", OUTPUT)


70 patients, 29330 rows -> C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\ModelingOptimization\AppliedModelingOptimization\Ahmad\assets\dataset\Diabetes-Data\diabetes_all_patients_with_codes.csv


In [9]:
import re
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../assets/dataset/Diabetes-Data").resolve()
OUTPUT = DATA_DIR / "diabetes_all_patients.csv"

CODE_MEANINGS = {
    "33": "Regular insulin dose",
    "34": "NPH insulin dose",
    "35": "UltraLente insulin dose",
    "48": "Unspecified blood glucose measurement",
    "57": "Unspecified blood glucose measurement",
    "58": "Pre-breakfast blood glucose measurement",
    "59": "Post-breakfast blood glucose measurement",
    "60": "Pre-lunch blood glucose measurement",
    "61": "Post-lunch blood glucose measurement",
    "62": "Pre-supper blood glucose measurement",
    "63": "Post-supper blood glucose measurement",
    "64": "Pre-snack blood glucose measurement",
    "65": "Hypoglycemic symptoms",
    "66": "Typical meal ingestion",
    "67": "More-than-usual meal ingestion",
    "68": "Less-than-usual meal ingestion",
    "69": "Typical exercise activity",
    "70": "More-than-usual exercise activity",
    "71": "Less-than-usual exercise activity",
    "72": "Unspecified special event",
}

pattern = re.compile(r"^data-(\d+)$")

chunks = []
for path in sorted(DATA_DIR.iterdir()):
    m = pattern.match(path.name)
    if not m or not path.is_file():
        continue

    patient_id = int(m.group(1))

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["Date", "Time", "Code", "Value"],
        dtype=str,
    )

    df.insert(0, "patientId", patient_id)

    # Assign sequential day numbers by unique date order within each patient file
    df["Day"] = pd.factorize(df["Date"])[0] + 1
    time_parsed = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce")
    df["MinuteOfDay"] = time_parsed.dt.hour * 60 + time_parsed.dt.minute


    df["CodeMeaning"] = df["Code"].map(CODE_MEANINGS).fillna("Unknown code")

    df = df.drop(columns=["Date", "Time"])
    df = df[["patientId", "Day", "MinuteOfDay", "Code", "Value", "CodeMeaning"]]

    chunks.append(df)

result = pd.concat(chunks, ignore_index=True)
result.to_csv(OUTPUT, index=False)

print(len(chunks), "patients,", len(result), "rows ->", OUTPUT)


70 patients, 29330 rows -> C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\ModelingOptimization\AppliedModelingOptimization\Ahmad\assets\dataset\Diabetes-Data\diabetes_all_patients.csv
